# Предсказание следующего товара
**Рекомендательная система на основе марковских переходов**

Подход: строим граф переходов между товарами и оцениваем вероятность P(j | i) — вероятность того, что после просмотра товара i пользователь просмотрит товар j.

## Импорты

In [ ]:
import json
import collections
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

## Шаг 0. Загрузка данных

In [ ]:
sessions = []
with open("sessions.jsonl") as f:
    for line in f:
        line = line.strip()
        if line:
            sessions.append(json.loads(line))

print(f"Всего сессий: {len(sessions)}")
print(f"Первая сессия: {sessions[0]}")
print(f"Вторая сессия: {sessions[1]}")

## Шаг 1. Анализ данных (EDA)

In [ ]:
# --- Базовая статистика ---
all_items = [item for session in sessions for item in session]
unique_items = set(all_items)
session_lengths = [len(s) for s in sessions]

print(f"Всего сессий:          {len(sessions)}")
print(f"Уникальных товаров:    {len(unique_items)}")
print(f"Всего взаимодействий: {len(all_items)}")
print(f"Средняя длина сессии:  {np.mean(session_lengths):.2f}")
print(f"Мин / Макс длина:      {min(session_lengths)} / {max(session_lengths)}")

# Повторные просмотры в одной сессии
sessions_with_repeats = sum(1 for s in sessions if len(s) != len(set(s)))
print(f"\nСессий с повторными товарами: {sessions_with_repeats} "
      f"({100*sessions_with_repeats/len(sessions):.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("EDA: обзор датасета", fontsize=14, fontweight="bold")

# 1. Распределение длин сессий
ax = axes[0]
length_counts = collections.Counter(session_lengths)
lengths = sorted(length_counts)
ax.bar(lengths, [length_counts[l] for l in lengths], color="steelblue", edgecolor="white")
ax.set_title("Распределение длин сессий")
ax.set_xlabel("Длина сессии")
ax.set_ylabel("Количество сессий")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# 2. Топ-20 самых популярных товаров
item_counts = collections.Counter(all_items)
top20 = item_counts.most_common(20)
ax = axes[1]
ax.barh([str(i) for i, _ in top20], [c for _, c in top20], color="coral")
ax.set_title("Топ-20 популярных товаров")
ax.set_xlabel("Частота")
ax.set_ylabel("ID товара")
ax.invert_yaxis()

# 3. Распределение частот товаров (хвост)
ax = axes[2]
freqs = sorted(item_counts.values(), reverse=True)
ax.plot(freqs, color="green")
ax.set_title("Частоты товаров (по убыванию)")
ax.set_xlabel("Ранг товара")
ax.set_ylabel("Частота")
ax.set_yscale("log")

plt.tight_layout()
plt.savefig("eda_plots.png", dpi=120, bbox_inches="tight")
plt.show()
print("График сохранён: eda_plots.png")

In [ ]:
# --- Дополнительно: редкие товары ---
once_items = sum(1 for v in item_counts.values() if v == 1)
print(f"Товаров, встретившихся ровно 1 раз: {once_items} "
      f"({100*once_items/len(unique_items):.1f}% от уникальных)")
print()
print("Наблюдения:")
print("- Распределение частот имеет длинный хвост (закон Ципфа):")
print("  несколько товаров очень популярны, большинство встречается редко.")
print("- Часть сессий содержит повторные просмотры одного товара —")
print("  это реалистично (пользователь возвращается к товару).")
print("- Самопереходы (i → i) нужно учесть при построении графа переходов.")

## Шаг 2. Train / Test Split

**Почему нельзя делать случайное разбиение?**

Данные — это последовательности во времени. Если перемешать элементы случайно, модель может «увидеть будущее»: обучиться на переходе `b → c`, а потом предсказывать `b`, когда в тесте стоит `a → b`. Это data leakage — утечка данных. Правильная схема: всегда обучаемся на начале последовательности, предсказываем конец.

In [ ]:
def train_test_split(
    sessions: list[list[int]],
) -> tuple[list[list[int]], list[int]]:
    """
    Разбиение сессий на train и test.
    Для каждой сессии:
      - все товары кроме последнего -> обучающая сессия
      - последний товар -> тестовая цель
    """
    train_sessions = [session[:-1] for session in sessions]
    test_targets   = [session[-1]  for session in sessions]
    return train_sessions, test_targets


train_sessions, test_targets = train_test_split(sessions)

print(f"Обучающих сессий:  {len(train_sessions)}")
print(f"Тестовых целей:    {len(test_targets)}")
print()
print(f"Пример сессии:     {sessions[0]}")
print(f"Train часть:       {train_sessions[0]}")
print(f"Test цель:         {test_targets[0]}")

## Шаг 3. Граф переходов и вероятности P(j | i)

In [ ]:
def build_transition_counts(
    train_sessions: list[list[int]],
) -> dict[int, collections.Counter]:
    """
    Строит граф переходов: transition_counts[i][j] = 
    сколько раз после товара i был просмотрен товар j.
    Самопереходы (i -> i) исключаются: они не несут
    информации о следующем *другом* товаре.
    """
    transition_counts = collections.defaultdict(collections.Counter)
    for session in train_sessions:
        for i in range(len(session) - 1):
            current = session[i]
            next_item = session[i + 1]
            if current != next_item:  # исключаем самопереходы
                transition_counts[current][next_item] += 1
    return transition_counts


def build_transition_probs(
    transition_counts: dict[int, collections.Counter],
) -> dict[int, dict[int, float]]:
    """
    Вычисляет P(j | i) = count(i->j) / sum_j count(i->j).
    Результат: transition_probs[i][j] = вероятность перехода i -> j.
    """
    transition_probs = {}
    for item, neighbors in transition_counts.items():
        total = sum(neighbors.values())
        transition_probs[item] = {j: cnt / total for j, cnt in neighbors.items()}
    return transition_probs


transition_counts = build_transition_counts(train_sessions)
transition_probs  = build_transition_probs(transition_counts)

print(f"Уникальных товаров в графе: {len(transition_probs)}")

# Сколько товаров вообще ни разу не стояли в позиции 'текущий'
items_in_graph = set(transition_probs.keys())
all_train_items = set(i for s in train_sessions for i in s)
cold_items = all_train_items - items_in_graph
print(f"Товаров без исходящих переходов (только последний в сессии "
      f"или только самопереходы): {len(cold_items)}")

# Пример
example_item = list(transition_probs.keys())[0]
top3 = sorted(transition_probs[example_item].items(), key=lambda x: -x[1])[:3]
print(f"\nПример: топ-3 перехода из товара {example_item}:")
for j, p in top3:
    print(f"  {example_item} -> {j}: P = {p:.3f}")

## Шаг 4. Рекомендательная модель

In [ ]:
def build_popularity_fallback(
    train_sessions: list[list[int]],
    k: int = 10,
) -> list[int]:
    """
    Возвращает топ-k самых популярных товаров по всему обучающему набору.
    Используется как fallback для холодных товаров.
    """
    item_counts = collections.Counter(
        item for session in train_sessions for item in session
    )
    return [item for item, _ in item_counts.most_common(k)]


def recommend(
    last_item: int,
    transition_probs: dict[int, dict[int, float]],
    popularity_fallback: list[int],
    k: int = 10,
) -> list[int]:
    """
    Возвращает топ-k рекомендаций для пользователя,
    последний просмотренный товар которого — last_item.

    Стратегия обработки граничных случаев:
    1. Если last_item есть в графе -> ранжируем по P(j | last_item),
       не включая сам last_item.
       Если переходов меньше k — добираем из popularity_fallback.
    2. Если last_item не встречался при обучении (cold-start) ->
       возвращаем popularity_fallback целиком.
    """
    if last_item not in transition_probs:
        # Cold-start: товар не видели при обучении
        return [item for item in popularity_fallback if item != last_item][:k]

    # Сортируем соседей по убыванию вероятности, исключаем сам last_item
    neighbors = transition_probs[last_item]
    ranked = sorted(
        ((j, p) for j, p in neighbors.items() if j != last_item),
        key=lambda x: -x[1],
    )
    recs = [j for j, _ in ranked[:k]]

    # Если рекомендаций меньше k — добираем из popularity_fallback
    if len(recs) < k:
        recs_set = set(recs)
        for item in popularity_fallback:
            if item not in recs_set and item != last_item:
                recs.append(item)
                recs_set.add(item)
            if len(recs) == k:
                break

    return recs


popularity_fallback = build_popularity_fallback(train_sessions, k=10)
print(f"Fallback (топ-10 популярных): {popularity_fallback}")

# Пример рекомендации
example_last = train_sessions[0][-1]
example_recs = recommend(example_last, transition_probs, popularity_fallback)
print(f"\nПоследний товар в сессии 0: {example_last}")
print(f"Топ-10 рекомендаций:        {example_recs}")

## Шаг 5. Оценка качества — Hit@10

In [ ]:
def hit_at_k(
    recommendations: list[list[int]],
    true_items: list[int],
    k: int = 10,
) -> float:
    """
    Вычисление Hit@K для списка предсказаний.
    Hit@K = доля примеров, где истинный товар попал в топ-K рекомендаций.
    """
    assert len(recommendations) == len(true_items), \
        "recommendations и true_items должны совпадать по длине"
    hits = 0
    for recs, true_item in zip(recommendations, true_items):
        if true_item in recs[:k]:
            hits += 1
    return hits / len(true_items)

In [ ]:
# --- Наша модель: Markov-переходы ---
markov_recs = [
    recommend(session[-1], transition_probs, popularity_fallback, k=10)
    for session in train_sessions
]
markov_hit = hit_at_k(markov_recs, test_targets, k=10)

# --- Бейзлайн: всегда топ-10 популярных ---
baseline_recs = [popularity_fallback[:10]] * len(test_targets)
baseline_hit  = hit_at_k(baseline_recs, test_targets, k=10)

print(f"Hit@10 — Markov-модель:  {markov_hit:.4f}  ({markov_hit*100:.1f}%)")
print(f"Hit@10 — Бейзлайн (топ-popular): {baseline_hit:.4f}  ({baseline_hit*100:.1f}%)")
print()
if markov_hit > baseline_hit:
    lift = (markov_hit - baseline_hit) / baseline_hit * 100
    print(f"Markov-модель лучше бейзлайна на {lift:.1f}%")
else:
    print("Markov-модель не превзошла бейзлайн — разбираемся ниже.")

In [ ]:
# --- Диаграмма сравнения ---
fig, ax = plt.subplots(figsize=(6, 4))
models = ["Бейзлайн\n(топ-10 популярных)", "Markov-модель\n(граф переходов)"]
scores = [baseline_hit, markov_hit]
colors = ["#b0c4de", "#2e8b57"]
bars = ax.bar(models, scores, color=colors, width=0.4, edgecolor="white")
for bar, score in zip(bars, scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f"{score:.3f}",
        ha="center", fontsize=12, fontweight="bold"
    )
ax.set_ylim(0, max(scores) * 1.3)
ax.set_ylabel("Hit@10")
ax.set_title("Сравнение моделей")
plt.tight_layout()
plt.savefig("comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("График сохранён: comparison.png")

## Шаг 6. Анализ результатов и рассуждения

In [ ]:
# Сколько тестовых целей вообще есть в обучающих данных?
train_items_set = set(i for s in train_sessions for i in s)
targets_seen = sum(1 for t in test_targets if t in train_items_set)
print(f"Тестовых целей, виденных при обучении: "
      f"{targets_seen}/{len(test_targets)} ({100*targets_seen/len(test_targets):.1f}%)")

# Сколько последних train-товаров имеют исходящие переходы?
last_train_items = [s[-1] for s in train_sessions]
has_transitions = sum(1 for i in last_train_items if i in transition_probs)
print(f"Последних train-товаров с исходящими переходами: "
      f"{has_transitions}/{len(last_train_items)} ({100*has_transitions/len(last_train_items):.1f}%)")

# Среднее число кандидатов в графе для последнего товара
num_neighbors = [
    len(transition_probs[i]) for i in last_train_items if i in transition_probs
]
print(f"\nСреднее кол-во соседей у 'последнего' товара: {np.mean(num_neighbors):.1f}")
print(f"Медиана: {np.median(num_neighbors):.1f}")

In [ ]:
print("=" * 55)
print("ИТОГОВЫЙ ВЫВОД")
print("=" * 55)
print()
print(f"  Markov Hit@10:   {markov_hit:.4f}")
print(f"  Baseline Hit@10: {baseline_hit:.4f}")
print()
print("Что это значит:")
print()
print("Markov-модель использует локальный контекст перехода:")
print("зная последний товар i, она смотрит, куда чаще всего")
print("переходили из i. Это работает, когда:")
print("  - пространство товаров невелико,")
print("  - переходы устойчивые (пользователи склонны")
print("    просматривать похожие товары подряд).")
print()
print("Бейзлайн (топ-популярные) работает хорошо, когда")
print("распределение просмотров сильно сконцентрировано")
print("на небольшом числе товаров (закон Ципфа).")
print()
print("В реальных данных Markov-модель, как правило, даёт")
print("лучшие результаты за счёт персонализации контекста.")
print()
print("Ограничения модели:")
print("  - Учитывается только последний товар (без истории).")
print("  - Холодные товары обрабатываются через popularity fallback.")
print("  - Самопереходы (i->i) исключены из графа.")